# GitHub Copilot Cost & Usage Analysis

This notebook retrieves and analyzes GitHub Copilot billing and usage data from the GitHub API to:
- Track seat allocations and costs
- Identify active vs. inactive seats
- Calculate cost efficiency metrics
- Support cost optimization decisions

**Data Source:** GitHub REST API (Copilot Billing endpoints)  
**Last Updated:** June 2026

## Parameters

Configure the notebook parameters using Fabric Variable Library for reusability across environments.

In [ ]:
# Import required libraries
import polars as pl
import requests
from datetime import datetime, timedelta
from typing import Dict, List, Optional
import json

In [ ]:
# Retrieve parameters from Fabric Variable Library
GITHUB_ORG = VariableLib.get("github_org")  # Your GitHub organization name
GITHUB_TOKEN = VariableLib.get("github_token")  # GitHub PAT with manage_billing:copilot or read:org scope
COPILOT_SEAT_COST = float(VariableLib.get("copilot_seat_cost", "19.00"))  # USD per seat per month

# API Configuration
# For GitHub Enterprise Server, set github_api_base to: https://your-enterprise.com/api/v3
# For GitHub.com or Enterprise Cloud, use default: https://api.github.com
GITHUB_API_BASE = VariableLib.get("github_api_base", "https://api.github.com")
GITHUB_API_VERSION = "2022-11-28"

# Display configuration (hide sensitive data)
print(f"API Base URL: {GITHUB_API_BASE}")
print(f"Organization: {GITHUB_ORG if GITHUB_ORG else '[NOT SET]'}")
print(f"Token configured: {'Yes' if GITHUB_TOKEN else 'No'}")
print(f"Seat cost: ${COPILOT_SEAT_COST}/month")

## Setup

Import required libraries and configure the API client.

In [ ]:
# Configure request headers for GitHub API
headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": GITHUB_API_VERSION
}

def make_github_request(endpoint: str, params: Optional[Dict] = None) -> Dict:
    """
    Make authenticated request to GitHub API with error handling.
    
    Args:
        endpoint: API endpoint path (e.g., '/orgs/{org}/copilot/billing')
        params: Optional query parameters
    
    Returns:
        JSON response as dictionary
    """
    url = f"{GITHUB_API_BASE}{endpoint}"
    response = requests.get(url, headers=headers, params=params)
    
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"API Error {response.status_code}: {response.text}")

print("✓ API client configured")

## Verify Permissions

Check your organization membership and role before proceeding.

## 1. Retrieve Billing Overview

Fetch high-level billing information including total seats, active/inactive counts, and settings.

In [ ]:
# Retrieve billing summary
billing_endpoint = f"/orgs/{GITHUB_ORG}/copilot/billing"
billing_data = make_github_request(billing_endpoint)

# Extract key metrics
seat_breakdown = billing_data.get("seat_breakdown", {})
total_seats = seat_breakdown.get("total", 0)
active_seats = seat_breakdown.get("active_this_cycle", 0)
inactive_seats = seat_breakdown.get("inactive_this_cycle", 0)
seats_added = seat_breakdown.get("added_this_cycle", 0)

# Calculate costs
monthly_cost = total_seats * COPILOT_SEAT_COST
cost_per_active_seat = monthly_cost / active_seats if active_seats > 0 else 0
utilization_rate = (active_seats / total_seats * 100) if total_seats > 0 else 0

# Display summary
print("=" * 60)
print("GITHUB COPILOT BILLING SUMMARY")
print("=" * 60)
print(f"\n📊 Seat Allocation:")
print(f"   Total Seats:        {total_seats:>6}")
print(f"   Active This Cycle:  {active_seats:>6}")
print(f"   Inactive:           {inactive_seats:>6}")
print(f"   Added This Cycle:   {seats_added:>6}")
print(f"\n💰 Cost Analysis:")
print(f"   Monthly Cost:       ${monthly_cost:>8,.2f}")
print(f"   Cost per Seat:      ${COPILOT_SEAT_COST:>8,.2f}")
print(f"   Cost per Active:    ${cost_per_active_seat:>8,.2f}")
print(f"\n📈 Utilization:")
print(f"   Active Rate:        {utilization_rate:>7.1f}%")
print(f"\n🔧 Plan Type: {billing_data.get('plan_type', 'N/A').upper()}")
print("=" * 60)

## 2. Retrieve Detailed Seat Assignments

Fetch individual seat assignments with user details and activity timestamps to identify optimization opportunities.

In [ ]:
# Retrieve all seat assignments (handle pagination)
seats_endpoint = f"/orgs/{GITHUB_ORG}/copilot/billing/seats"
all_seats = []
page = 1
per_page = 100

print("Fetching seat assignments...")
while True:
    params = {"page": page, "per_page": per_page}
    response = make_github_request(seats_endpoint, params)
    
    seats = response.get("seats", [])
    if not seats:
        break
    
    all_seats.extend(seats)
    print(f"  Retrieved {len(all_seats)} seats...", end="\r")
    
    # Check if there are more pages
    if len(seats) < per_page:
        break
    page += 1

print(f"✓ Retrieved {len(all_seats)} total seat assignments")

# Convert to Polars DataFrame
seats_data = []
for seat in all_seats:
    assignee = seat.get("assignee", {})
    seats_data.append({
        "user_login": assignee.get("login"),
        "user_name": assignee.get("name"),
        "user_email": assignee.get("email"),
        "created_at": seat.get("created_at"),
        "updated_at": seat.get("updated_at"),
        "last_activity_at": seat.get("last_activity_at"),
        "last_activity_editor": seat.get("last_activity_editor"),
        "pending_cancellation_date": seat.get("pending_cancellation_date"),
        "assigning_team": seat.get("assigning_team", {}).get("name") if seat.get("assigning_team") else None
    })

df_seats = pl.DataFrame(seats_data)

# Convert timestamp columns to datetime
timestamp_cols = ["created_at", "updated_at", "last_activity_at", "pending_cancellation_date"]
for col in timestamp_cols:
    if col in df_seats.columns:
        df_seats = df_seats.with_columns(
            pl.col(col).str.to_datetime().alias(col)
        )

print(f"\n✓ Created DataFrame with {df_seats.height} rows × {df_seats.width} columns")
df_seats.head()

## 3. Analyze Seat Activity

Identify inactive seats and calculate potential cost savings from optimization.

In [ ]:
# Calculate activity metrics
current_date = datetime.now()
threshold_30_days = current_date - timedelta(days=30)
threshold_60_days = current_date - timedelta(days=60)
threshold_90_days = current_date - timedelta(days=90)

# Add activity classification columns
df_activity = df_seats.with_columns([
    # Days since last activity
    pl.when(pl.col("last_activity_at").is_not_null())
    .then((pl.lit(current_date) - pl.col("last_activity_at")).dt.total_days())
    .otherwise(None)
    .alias("days_since_activity"),
    
    # Activity status
    pl.when(pl.col("last_activity_at").is_null())
    .then(pl.lit("Never Active"))
    .when(pl.col("last_activity_at") >= threshold_30_days)
    .then(pl.lit("Active (< 30 days)"))
    .when(pl.col("last_activity_at") >= threshold_60_days)
    .then(pl.lit("Inactive (30-60 days)"))
    .when(pl.col("last_activity_at") >= threshold_90_days)
    .then(pl.lit("Inactive (60-90 days)"))
    .otherwise(pl.lit("Inactive (> 90 days)"))
    .alias("activity_status"),
    
    # Cost per seat
    pl.lit(COPILOT_SEAT_COST).alias("monthly_cost")
])

# Aggregate by activity status
activity_summary = (
    df_activity
    .group_by("activity_status")
    .agg([
        pl.count().alias("seat_count"),
        pl.sum("monthly_cost").alias("total_cost")
    ])
    .sort("seat_count", descending=True)
)

print("\n" + "=" * 60)
print("SEAT ACTIVITY ANALYSIS")
print("=" * 60)
print(activity_summary)

# Calculate potential savings from inactive seats
inactive_seats_count = df_activity.filter(
    pl.col("activity_status").str.contains("Inactive|Never Active")
).height
potential_savings = inactive_seats_count * COPILOT_SEAT_COST

print(f"\n💡 Optimization Opportunity:")
print(f"   Inactive Seats:     {inactive_seats_count}")
print(f"   Potential Savings:  ${potential_savings:,.2f}/month")
print("=" * 60)

## 4. Identify Inactive Users

List users who haven't used Copilot in 60+ days for potential seat reclamation.

In [ ]:
# Filter for users inactive for 60+ days
df_inactive = (
    df_activity
    .filter(
        (pl.col("days_since_activity") > 60) | 
        (pl.col("last_activity_at").is_null())
    )
    .select([
        "user_login",
        "user_name",
        "last_activity_at",
        "days_since_activity",
        "last_activity_editor",
        "assigning_team",
        "activity_status",
        "monthly_cost"
    ])
    .sort("days_since_activity", descending=True, nulls_last=False)
)

print(f"\n🔍 Found {df_inactive.height} inactive users (60+ days or never active)")
print(f"💰 Monthly cost: ${df_inactive.height * COPILOT_SEAT_COST:,.2f}\n")

# Display inactive users
df_inactive.head(20)

## 5. Editor Usage Distribution

Analyze which editors are being used to understand tooling preferences.

In [ ]:
# Analyze editor distribution (only for users with activity)
df_editor_usage = (
    df_activity
    .filter(pl.col("last_activity_editor").is_not_null())
    .group_by("last_activity_editor")
    .agg([
        pl.count().alias("user_count"),
        (pl.count() * COPILOT_SEAT_COST).alias("monthly_cost")
    ])
    .with_columns([
        (pl.col("user_count") / pl.col("user_count").sum() * 100).alias("percentage")
    ])
    .sort("user_count", descending=True)
)

print("\n" + "=" * 60)
print("EDITOR USAGE DISTRIBUTION")
print("=" * 60)
print(df_editor_usage)
print("=" * 60)

## 6. Export Data for Reporting

Save the processed data to Delta Lake format for further analysis and reporting.

In [ ]:
# Define output paths (adjust for your Fabric lakehouse)
# output_base = "Tables/github_copilot"
# 
# # Add snapshot timestamp
# snapshot_date = datetime.now().strftime("%Y-%m-%d")
# 
# df_activity_with_snapshot = df_activity.with_columns([
#     pl.lit(snapshot_date).alias("snapshot_date"),
#     pl.lit(GITHUB_ORG).alias("organization")
# ])
# 
# # Write to Delta Lake
# df_activity_with_snapshot.write_delta(
#     f"{output_base}/seat_activity",
#     mode="append"
# )
# 
# print(f"✓ Data exported to {output_base}/seat_activity")
# print(f"  Rows written: {df_activity_with_snapshot.height}")
# print(f"  Snapshot date: {snapshot_date}")

print("📝 Export code ready (uncomment and configure output path)")

## Summary & Recommendations

Review the cost analysis and identify actionable optimization steps.

In [ ]:
# Generate summary report
never_active = df_activity.filter(pl.col("last_activity_at").is_null()).height
inactive_30_60 = df_activity.filter(
    (pl.col("days_since_activity") > 30) & (pl.col("days_since_activity") <= 60)
).height
inactive_60_plus = df_activity.filter(pl.col("days_since_activity") > 60).height

print("\n" + "=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print(f"\n📊 Current State:")
print(f"   Total Seats:              {total_seats}")
print(f"   Monthly Cost:             ${monthly_cost:,.2f}")
print(f"   Utilization Rate:         {utilization_rate:.1f}%")
print(f"\n⚠️  Inactive Seats Breakdown:")
print(f"   Never Active:             {never_active} (${never_active * COPILOT_SEAT_COST:,.2f}/mo)")
print(f"   Inactive 30-60 days:      {inactive_30_60} (${inactive_30_60 * COPILOT_SEAT_COST:,.2f}/mo)")
print(f"   Inactive 60+ days:        {inactive_60_plus} (${inactive_60_plus * COPILOT_SEAT_COST:,.2f}/mo)")
print(f"\n💡 Potential Monthly Savings: ${potential_savings:,.2f}")
print(f"   Annual Impact:             ${potential_savings * 12:,.2f}")
print(f"\n📋 Recommended Actions:")
print(f"   1. Review {never_active} users who never activated Copilot")
print(f"   2. Contact {inactive_60_plus} users inactive for 60+ days")
print(f"   3. Consider reclaiming inactive seats quarterly")
print(f"   4. Implement usage monitoring and automated alerts")
print("=" * 60)

In [ ]:
# Check authenticated user and permissions
try:
    # Get authenticated user
    user_response = make_github_request("/user")
    print(f"✓ Authenticated as: {user_response.get('login')}")
    
    # Check organization membership
    membership_response = make_github_request(f"/orgs/{GITHUB_ORG}/memberships/{user_response.get('login')}")
    role = membership_response.get('role')
    state = membership_response.get('state')
    
    print(f"✓ Organization: {GITHUB_ORG}")
    print(f"✓ Your role: {role}")
    print(f"✓ Membership state: {state}")
    
    # Check if user can access billing
    if role in ['admin', 'billing_manager']:
        print("\n✅ You have access to billing information")
    else:
        print("\n⚠️  Warning: You may not have access to billing endpoints")
        print("   Required role: 'admin' (owner) or 'billing_manager'")
        
except Exception as e:
    print(f"❌ Error checking permissions: {str(e)}")
    print("   This might indicate insufficient permissions or incorrect token/org configuration")